# Agentic Workflows as Probabilistic Programs

LLMs are increasingly being used as agents when it comes to solving problems and tasks. These scenarios involve single or multiple LLMs interacting with the environment via tools and gathering information in order to accomplish a task or to find answer to a complex query. There are several workflows that people have come up with based on specific tasks and tools available for LLMs to use. Some notable examples include Self-Refine, Reflexion, ReAct, Magentic-One etc.

Since language models are generative, we envision these workflows to be probabilistic programs. With this perspective, we aim to separate the "what" from "how", the declarative specification of the task from how it is achieved. Now, which components of these workflows belong in "what" and which components belong in "how" is unfortunately subjective and the right choice depends on which separation opens the most uses. One perspective where everything other than the query and the final answer belongs in "how" is described in this [overleaf document](https://www.overleaf.com/read/vkkvczrnnkwb#e0fe1e)

In this note, we focus on a different perspective where a few more components than the query and the final answer make it to "what". This is primarily done to allow reasoning about a wider set of components of these agentic workflows.

## ReAct

A typical ReAct pipeline sends an LLM two prompts 

1) A system prompt that tells you what sequence of steps to follow 
2) A task prompt describing the query at hand. Upon receiving these prompts, the LLM goes through a loop of outputting thoughts and code blocks which are executed, appended to the prompt and the loop continues until final_answer is reached.

Algorithmically, it looks something like below:

```python
system_prompt = …
task = …
is_final = False
memory = task + system_prompt
while not is_final:
	thought, action = sample(LLM(.. | memory))
	is_final = check(action)
	if is_final:
		answer = extract_answer(action)
	else:
		observation = exec(action)
		memory += observation
return answer
```

The above code is currently implemented as CodeAgent in smolagents library.



This [notebook](playground/fun.ipynb) uses CodeAgent in smolagent library to carry out a ReAct loop on the first 20 examples of GAIA benchmark. It shows 20% accuracy which varies significantly from run to run and uses on average 229,556.85 tokens per question which is a lot since the limit of context window for OpenAI API is 128,000.

## Token discrepancies

It can be confusing in smolagents that how are the statistics about input and output tokens are being generated. Atleast for OpenAI model, these token statistics printed by the library are cumulative input and output tokens totaled over all the API calls made for a query till a particular point. This differs from the length of the context passed to the single call to API.

Also, note that OpenAI APIs perform [**prompt caching**](https://developers.openai.com/api/docs/guides/prompt-caching) which can affect how the price of the tokens differ.

## Baselines

Before we move on, let's see how different LLMs perform as backbones for ReAct. 

**Full validation set (165 questions):**

- The following list does not include deepseek-ai/DeepSeek-R1 because it is being deprecated and not available for serverless use on TogetherAI.
- The Qwen models are being used with `{"enable_thinking": False}` as a parameter.

In [3]:
import pickle
from pathlib import Path

import pandas as pd

BASELINE_DIR = Path("baseline")

# Read from the per-example .pkl cache dirs rather than the .jsonl logs: evaluate_agent
# writes pickle_dir/{i}.pkl once per example index, overwritten in place on recompute, so
# each file always reflects that example's single latest result -- no de-duplication needed,
# unlike the .jsonl logs which are append-only across every run/restart across sessions.
MODEL_DIRS = {
    "gpt-4o": BASELINE_DIR / "naive_react_gpt-4o",
    "gpt-5.4-mini": BASELINE_DIR / "naive_react_gpt-5.4-mini",
    "Qwen3.7-Plus": BASELINE_DIR / "naive_react_Qwen" / "Qwen3.7-Plus_False",
    "Qwen3.5-9B": BASELINE_DIR / "naive_react_Qwen" / "Qwen3.5-9B_False",
}


def summarize_results(pkl_dir: Path):
    records = []
    for pkl_file in sorted(pkl_dir.glob("*.pkl"), key=lambda p: int(p.stem)):
        with open(pkl_file, "rb") as f:
            row = pickle.load(f)
        token_counts = row.get("token_counts") or {}
        records.append({
            "is_correct": bool(row.get("is_correct", False)),
            "num_steps": row.get("num_steps", 0),
            "total_tokens": token_counts.get("total_tokens", 0),
            "error": row.get("error"),
        })

    n = len(records)
    if n == 0:
        return None
    correct = sum(r["is_correct"] for r in records)
    return {
        "n": n,
        "correct": correct,
        "accuracy": correct / n,
        "avg_steps": sum(r["num_steps"] for r in records) / n,
        "avg_tokens": sum(r["total_tokens"] for r in records) / n,
        "errors": sum(1 for r in records if r["error"]),
    }


rows = []
for model_name, pkl_dir in MODEL_DIRS.items():
    if not pkl_dir.exists():
        rows.append({"Model": model_name, "Accuracy": "missing dir", "Questions": 0, "Avg Steps": "-", "Avg Tokens": "-", "Errors": "-"})
        continue
    stats = summarize_results(pkl_dir)
    if stats is None:
        rows.append({"Model": model_name, "Accuracy": "no data", "Questions": 0, "Avg Steps": "-", "Avg Tokens": "-", "Errors": "-"})
        continue
    rows.append({
        "Model": model_name,
        "Accuracy": f"{stats['correct']}/{stats['n']} = {stats['accuracy']:.1%}",
        "Questions": stats["n"],
        "Avg Steps": f"{stats['avg_steps']:.1f}",
        "Avg Tokens": f"{stats['avg_tokens']:,.0f}",
        "Errors": stats["errors"],
    })

pd.DataFrame(rows)

,Model,Accuracy,Questions,Avg Steps,Avg Tokens,Errors
0,gpt-4o,43/165 = 26.1%,165,10.8,"136,605",0
1,gpt-5.4-mini,54/165 = 32.7%,165,5.4,"37,291",0
2,Qwen3.7-Plus,97/164 = 59.1%,164,19.4,"448,291",9
3,Qwen3.5-9B,82/164 = 50.0%,164,18.9,"463,114",2
